<h2>Problem Statement</h2>
With FIFA is in the blood of many people of the world. You are tasked to tell the story of unsung analysts who put great efforts to provide accurate data to answer every question of fans. The FIFA World Cup is a global football competition contested by the various football-playing nations of the world. It is contested every four years and is the most prestigious and important trophy in the sport of football. </br>
The World Cups dataset shows all information about all the World Cups in history, while the World Cup Matches dataset shows all the results from the matches contested as part of the cups. Find key metrics and factors that influence the World Cup win. Do your own research and come up with your findings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

wc=pd.read_csv("WorldCups.csv", index_col=0)
print(wc.head())

In [ ]:
wc_matches=pd.read_csv("WorldCupMatches.csv", index_col=0)
print(wc_matches.head())

<h2> Key metrics and factors influencing World Cup win </h2>


In [ ]:
# Countries with most Wins
winning_country_stats = wc['Winner'].value_counts()
print("Countries with most Wins:")
print(winning_country_stats)

In [ ]:
# 10 Most Successful Teams in Matches:
winning_teams = wc_matches.loc[wc_matches['Home Team Goals'] > wc_matches['Away Team Goals'], 'Home Team Name']
winning_teams = winning_teams.append(wc_matches.loc[wc_matches['Away Team Goals'] > wc_matches['Home Team Goals'], 'Away Team Name'])
winning_teams_count = winning_teams.value_counts()
print("Top 10 Most Successful Teams in Matches:")
print(winning_teams_count.sort_values(ascending=False).head(10))

In [ ]:
#checking if playing home games influences the win
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

X = wc_matches[['Home Team Goals', 'Away Team Goals']]
y = (wc_matches['Home Team Goals'] > wc_matches['Away Team Goals']).astype(int)

imputer = SimpleImputer(strategy='mean')  
X = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
clf = RandomForestClassifier(n_estimators=100, random_state=1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy}")

probability_home_team_winning = np.sum(y_test) / len(y_test)
print(f"Probability of a Home Team Winning: {probability_home_team_winning}")
probability_away_team_winning = 1 - probability_home_team_winning
print(f"Probability of a Away Team Winning: {probability_away_team_winning}")


In [ ]:
teams = ['Home Team', 'Away Team']
probabilities = [probability_home_team_winning, probability_away_team_winning]

plt.bar(teams, probabilities, color=['blue', 'red'])
plt.title('Probability of Home Team Winning vs. Away Team Winning')
plt.ylabel('Probability')
plt.show()

print(f"\n\nAnalysis \nSince the Probability of a Away Team Winning ({probability_away_team_winning}) is  much more than the Probability of a Home Team Winning({probability_home_team_winning}), we can conclude that playing home games is not a factor influencing the win " )


In [ ]:
categorical_columns = ['Stage', 'Stadium', 'City', 'Home Team Name', 'Away Team Name',
                        'Win conditions', 'Referee', 'Assistant 1', 'Assistant 2',
                        'Home Team Initials', 'Away Team Initials']

# One-hot encode categorical columns
X_train_encoded = pd.get_dummies(X_train, columns=categorical_columns)

nan_check = X_train_encoded.isnull().sum()


X_train_encoded.fillna(X_train_encoded.mean(), inplace=True)

X_train_encoded, X_test_encoded, y_train, y_test = train_test_split(X_train_encoded, y_train, test_size=0.2, random_state=42)

# Training the Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_encoded, y_train)

feature_importance = clf.feature_importances_

print("\nFeature Importance:")
for feature, importance in zip(X_train_encoded.columns, feature_importance):
    print(f"{feature}: {importance}")


In [ ]:
feature_importance_tuples = list(zip(X_train_encoded.columns, feature_importance))
sorted_feature_importance = sorted(feature_importance_tuples, key=lambda x: x[1], reverse=True)

# Print the top 5 most important features
print("\nTop 5 Most Important Features:")
for feature, importance in sorted_feature_importance[:5]:
    print(f"{feature}: {importance}")


In [ ]:
import numpy as np
import pandas as pd

wc=pd.read_csv("WorldCups.csv", index_col=0)
print(wc.head())

In [ ]:
wc_matches=pd.read_csv("WorldCupMatches.csv", index_col=0)
print(wc_matches.head())

In [ ]:
# Countries with most Wins
winning_country_stats = wc['Winner'].value_counts()
print("Countries with most Wins:")
print(winning_country_stats)

In [ ]:
# 10 Most Successful Teams in Matches:
winning_teams = wc_matches.loc[wc_matches['Home Team Goals'] > wc_matches['Away Team Goals'], 'Home Team Name']
winning_teams = winning_teams.append(wc_matches.loc[wc_matches['Away Team Goals'] > wc_matches['Home Team Goals'], 'Away Team Name'])
winning_teams_count = winning_teams.value_counts()
print("Top 10 Most Successful Teams in Matches:")
print(winning_teams_count.sort_values(ascending=False).head(10))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix



# whether the home team won or not
wc_matches['HomeTeamWon'] = (wc_matches['Home Team Goals'] > wc_matches['Away Team Goals']).astype(int)

features=['Home Team Goals', 'Away Team Goals']
X = wc_matches[features]
y = wc_matches['HomeTeamWon']

# Imputing missing values
imputer = SimpleImputer(strategy='mean')  
X = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
feature_importance = clf.feature_importances_

print("\nFeature Importance:")
for feature, importance in zip(features, feature_importance):
    print(f"{feature}: {importance}")

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("\nModel Accuracy:", accuracy)

# Check the probability of a home team winning
home_team_goals = 0  # You can adjust this value
away_team_goals = 0  # You can adjust this value
probability_home_team_wins = clf.predict_proba([[home_team_goals, away_team_goals]])[0][1]




# Evaluate the model
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
confusion_mat = confusion_matrix(y_test, y_pred)

print("\nModel Accuracy:", accuracy)
print("\nConfusion Matrix:\n", confusion_mat)




home_team_goals = 2  # You can adjust this value
away_team_goals = 1  # You can adjust this value
total_goals = home_team_goals + away_team_goals

probability_home_team_wins = clf.predict_proba([[home_team_goals, away_team_goals]])[0][1]
adjusted_probability = home_team_goals / total_goals

print(" Probability of a Home Team Winning:", adjusted_probability)



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Away Team Wins', 'Home Team Wins'], yticklabels=['Away Team Wins', 'Home Team Wins'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()